# Training Data Cleaning for [BirdCLEF2023](https://www.kaggle.com/competitions/birdclef-2023)
## Introduction
This notebook looks at the data from BirdCLEF 2021, 2022 and 2023.  Produces two outputs

- A csv file with the 2023 dataset reduced to 'primary_label', 'secondary_labels', 'type', 'filename','filepath'.  File path is relative to the kaggle input folder.
- A csv file in the same format, with rows containing 100% of the 2023 data, plus training samples from 22 & 21, without any duplicates.

All the file names in both csv files are now unique, and in the same format. I have removed 6,686 cases where there was overlap between competitions, plus a remaining 7 cases where the same file name existed in more than one class folder.

## Usage
You could link the datasets from all three years, and use these two CSV files to create your training and validation splits, for pretraining on all the data and finetuning on the 2023 dataset, all in the same format.

## Dataset
I have uploaded the CSV files to a dataset [here](https://www.kaggle.com/datasets/ollypowell/cleaned-training-labels-21-23-for-birdclef2023)

In [ ]:
import pandas as pd
import librosa
from pathlib import Path
from collections import Counter
from IPython.display import Audio

In [ ]:
# setup file paths
data_folder = Path('/kaggle/input')  #modify to suit
out_folder = Path('/kaggle/working')

train_21 = data_folder / 'birdclef-2021' / 'train_short_audio'
train_22 =  data_folder / 'birdclef-2022' / 'train_audio'
train_23 = data_folder / 'birdclef-2023' / 'train_audio'

train_21_csv = data_folder / 'birdclef-2021' / 'train_metadata.csv'
train_22_csv = data_folder / 'birdclef-2022' / 'train_metadata.csv'
train_23_csv = data_folder / 'birdclef-2023' / 'train_metadata.csv'

in_csv_fields = ['primary_label', 'secondary_labels', 'type', 'filename']
csv_datatypes = {'primary_label': str, 'secondary_labels': str, 'type': str, 'filename': str}

out_csv_all_pth = out_folder / 'train_21_22_23.csv'
out_csv_23_pth = out_folder / 'train_23.csv'

In [ ]:
def play_audio(file_path):
    audio_abe, sr_abe = librosa.load(file_path)
    return Audio(data=audio_abe, rate=sr_abe)

In [ ]:
sfiles_23 = {f:str(f.name) for f in Path(train_23).rglob('*.ogg')}
sfiles_22 = {f:str(f.name) for f in Path(train_22).rglob('*.ogg')}
sfiles_21 = {f:str(f.name) for f in Path(train_21).rglob('*.ogg')}

all_sfiles = {**sfiles_23, **sfiles_22, **sfiles_21}

unique_names =  set(all_sfiles.values())
unique_names_w_folder = set([str(k.parent.name) + '/' + str(k.name) for (k,v) in all_sfiles.items()])
len_all_sfiles = len(all_sfiles)
print(f'All file names by path: {len_all_sfiles}')
print(f'Unique file names including folder name: {len(unique_names_w_folder)}')
print(f'Unique file names {len(unique_names)}')

It would be good to varify that the files are genuinely different birds, not just different spellings of the folders, or files in the wrong folders.

In [ ]:
item_counts = Counter(all_sfiles.values()) 
duplicates = list([item for item in item_counts if item_counts[item]>1])
print(f'Number of duplicate file names found: {len(duplicates)}')

Now I'm going to examine the folder names for each of these duplicates

In [ ]:
dupes_dict = {}
for dup in duplicates:
    #find matching file paths and put them in a dict
    dupes_dict[dup] = [str(k.parent.name) + '/' + str(k.name) for (k,v) in all_sfiles.items() if v == dup]
dict(list(dupes_dict.items())[0: 6])

This is as expected, most duplicates have the same folder name, they are just overlap between different competitions.  But the first on the list does not, and there are potentially 5 others like this as it was earier noted there are 5 more unique folder-name combinations than just unique names.  I'm going to sort through the duplicates to find these examples.

In [ ]:
matches = {k:v for (k,v) in dupes_dict.items() if len(set(v)) > 1}
matches

11 Files found.  Now all that's left to do is listen to see if they are actually different birds, or just labelling errors

In [ ]:
matches_dict = {}
for key in matches.keys():
    matches_dict[key] = [k for (k,v) in all_sfiles.items() if v == key]
matches_dict

Going through the four pairs and one triple, and checking if they are actually different files.

In [ ]:
# Checking XC316684.ogg gnbcam2
play_audio('/kaggle/input/birdclef-2021/train_short_audio/mallar3/XC501149.ogg')

In [ ]:
# Checking XC316684.ogg cibwar1
play_audio('/kaggle/input/birdclef-2023/train_audio/cibwar1/XC316684.ogg')

They are definately the same file.  Sounds like there are two types of birds present.

In [ ]:
# Checking XC514027.ogg gnbcam2
play_audio('/kaggle/input/birdclef-2023/train_audio/laudov1/XC514027.ogg')

In [ ]:
# Checking XC514027.ogg eucdov
play_audio('/kaggle/input/birdclef-2021/train_short_audio/eucdov/XC514027.ogg')

They are definately the same file. There are at least two different birds present.

In [ ]:
# Checking  XC294370.ogg brant
play_audio('/kaggle/input/birdclef-2022/train_audio/brant/XC294370.ogg')

In [ ]:
# Checking  XC294370.ogg gadwal
play_audio('/kaggle/input/birdclef-2022/train_audio/gadwal/XC294370.ogg')

I could only hear one bird there, but it's a really short clip

In [ ]:
# XC477595.ogg  norcar
play_audio('/kaggle/input/birdclef-2022/train_audio/norcar/XC477595.ogg')

In [ ]:
# XC477595.ogg
play_audio('/kaggle/input/birdclef-2021/train_short_audio/carwre/XC477595.ogg')

The same file, and definately at least two birds present.

In [ ]:
# XC501149.ogg  gnwtea
play_audio('/kaggle/input/birdclef-2022/train_audio/gnwtea/XC501149.ogg')

In [ ]:
#XC501149.ogg mallar3
play_audio('/kaggle/input/birdclef-2021/train_short_audio/mallar3/XC501149.ogg')

Not sure about that one, difinately the same file, not sure if there was more than one type of bird.

So in summary 3/5 of these pairs had two birds at least present, so they could be correct labels.  The other two I'm not sure about.  However only three files: cibwar1/XC316684.ogg, gnbcam2/XC316684.ogg, and eucdov/XC514027.ogg are in this years competition, and they all have more than one bird present, so could their classification could be correct. The rest could only be useful for pre-training, so they won't make much difference anyway.  I'm going to drop everything except cibwar1/XC316684.ogg, gnbcam2/XC316684.ogg & eucdov/XC514027.ogg

In [ ]:
keep_files = ['/kaggle/input/birdclef-2023/train_audio/cibwar1/XC316684.ogg', 
              '/kaggle/input/birdclef-2023/train_audio/gnbcam2/XC316684.ogg',
              '/kaggle/input/birdclef-2021/train_short_audio/eucdov/XC514027.ogg']
drop_files =  [item for sublist in matches_dict.values() for item in sublist if str(item) not in keep_files]
len(drop_files)  

With the duplicates understood, I'm going to re-build the combined dictionary from the three years, without duplicates

In [ ]:
sfiles_22 = {k:v for (k,v) in sfiles_22.items() if v not in sfiles_23.values()}
vals_22_23 = list(sfiles_23.values()) + list(sfiles_22.values())
sfiles_21 = {k:v for (k,v) in sfiles_21.items() if v not in vals_22_23}

all_sfiles = {**sfiles_23, **sfiles_22, **sfiles_21}
all_sf_len = len(all_sfiles)
print(f'{all_sf_len} unique file names in the dict from 21, 22, 23')

In [ ]:
# And drop those files from 22 & 21 that might be mislabeled, or have two birds
for key in drop_files:
    all_sfiles.pop(key, None)
    
print(f'{len(all_sfiles)} unique file names in the dict from 21, 22, 23')
print(f'{all_sf_len - len(all_sfiles)} items were removed because they were the same file name in different class folders')

So now I have two cleaner dictionaries in the form {(file_path: file_name)}  `sfiles_23` with the 2023 data and all_sfiles with unique data from 21, 22 & 23.

In [ ]:
df_23 = pd.read_csv(train_23_csv,  usecols=in_csv_fields, dtype=csv_datatypes, index_col=None)
df_22 = pd.read_csv(train_22_csv,  usecols=in_csv_fields, dtype=csv_datatypes, index_col=None)
df_21 = pd.read_csv(train_21_csv,  usecols=in_csv_fields, dtype=csv_datatypes, index_col=None)

print(f'Dataframe lengths: {[len(df_23), len(df_22), len(df_21)]}')
print(f'Total number of sound files in the three datasets: {len_all_sfiles}')
print(f'Total rows (should match the number of files): {len(df_23) + len(df_22) +len(df_21)}')
df_23.head(3)

In [ ]:
df_22.head(3)

In [ ]:
df_21.head(3)

The dataframes are consistant, except for the filenames of 2021 which does not include the parent folder, and for matching all three years data I'll need the full file path.  I'm going to a new column for the full file path, then strip the parent from the filename column in 22 and 23.

In [ ]:
def build_path(bird,fn): 
    return str(train_21) + '/' + bird + '/' + fn

df_23['filepath'] = df_23['filename'].apply(lambda x: str(train_23) + '/' + x)
df_22['filepath'] = df_22['filename'].apply(lambda x: str(train_22) + '/' + x)
df_21['filepath'] = df_21.apply(lambda row: build_path(row['primary_label'], row['filename']), axis=1)

df_23['filename'] = df_23['filename'].apply(lambda x: x.split('/')[-1] )
df_22['filename'] = df_22['filename'].apply(lambda x: x.split('/')[-1] )

print(f'Length of 2023 dataframe: {len(df_23)}')
df_23.head(3)

In [ ]:
all_dfs = pd.concat([df_23, df_22, df_21])

print(f'Length of the combined dataframe: {len(all_dfs)}')
all_dfs.head(3)

Now I can use the dictionary of unique files paths to filter out all duplicates

In [ ]:
filter_list = [str(x) for x in all_sfiles.keys()]
all_dfs = all_dfs[all_dfs['filepath'].isin(filter_list)]

print(f'Length of 2023 dataframe: {len(df_23)}')
print(f'Length of the combined dataframe: {len(all_dfs)}')
print(f'Expected length from number of unique files: {len(all_sfiles)}')

In [ ]:
all_dfs.head()

In [ ]:
df_23.to_csv(out_csv_23_pth, index=False)
all_dfs.to_csv(out_csv_all_pth, index=False)